# Smart Meeting Follow-Up Agent

### Problem

After a meeting, someone has to read the notes, figure out the action items, work out who owns each one and by when, and make sure tasks get created and people get notified. This gets skipped more often than not.

### What this agent does

1. Takes raw meeting notes as plain text.
2. Extracts action items - task, owner, deadline - using an LLM.
3. Applies a hard guardrail: if an action item has no clear owner, the agent never guesses one. It is flagged for a human to assign.
4. Fills in a sensible default deadline (one week out) for owned items that don't have one.
5. Creates a task per owned action item via a tool, and sends a summary notification via a second tool.

This notebook is the exploration/dev version of the agent. A small Flask + HTML app implementing the same logic behind a simple UI is provided alongside it as three files: `index.html`, `app.py`, `requirements.txt`.

### Provider switch

This notebook can run against **either Boeing's internal `BoeingChatModel`** or **OpenAI's `ChatOpenAI`**. Only one provider needs valid credentials at a time. Set `LLM_PROVIDER` in Section 0 (or in your `.env` file) to `"boeing"` or `"openai"` to switch - no other code needs to change.


---
## SECTION 0 - Installation and Setup


In [ ]:
# ============================================================
# SECTION 0a - ENVIRONMENT SETUP (READ ME FIRST)
# Run these commands in your terminal (VS Code integrated
# terminal or otherwise), NOT inside this notebook cell.
# ============================================================

# ------------------------------------------------------------
# 1) CREATE A VIRTUAL ENVIRONMENT (run once, from your project folder)
# ------------------------------------------------------------
# Windows (PowerShell / cmd):
#     python -m venv venv
#     venv\Scripts\activate
#
# macOS / Linux:
#     python3 -m venv venv
#     source venv/bin/activate
#
# Then select this venv as the Jupyter kernel in VS Code:
#     Command Palette (Ctrl+Shift+P) -> "Python: Select Interpreter" -> ./venv
#     In the notebook: "Select Kernel" (top-right) -> the same venv.

# ------------------------------------------------------------
# 2) UPGRADE PIP
# ------------------------------------------------------------
# python -m pip install --upgrade pip

# ------------------------------------------------------------
# 3) INSTALL PACKAGES
# ------------------------------------------------------------
# Always required:
# pip install langchain-core python-dotenv
#
# Required ONLY if LLM_PROVIDER = "openai":
# pip install langchain-openai
#
# Required ONLY if LLM_PROVIDER = "boeing"
# (internal packages - install from your organization's package index):
# pip install boeing-chat-model boeing-embeddings

# ------------------------------------------------------------
# 4) CREATE A .env FILE (same folder as this notebook)
# ------------------------------------------------------------
# LLM_PROVIDER=boeing                                # or "openai"
#
# # Required if LLM_PROVIDER=boeing
# UDAL_PAT=your_udal_personal_access_token_here
#
# # Required if LLM_PROVIDER=openai
# OPENAI_API_KEY=sk-your-openai-key-here

# ------------------------------------------------------------
# 5) (Optional) FREEZE YOUR ENVIRONMENT
# ------------------------------------------------------------
# pip freeze > requirements.txt


In [ ]:
import subprocess, sys

# Always required
base_packages = ["langchain-core", "python-dotenv"]

print("[SETUP] Installing base dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + base_packages, check=True)
print("[SETUP] Base dependencies installed.")

# NOTE: install exactly one provider's package depending on LLM_PROVIDER below.
#
# OpenAI path:
# subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langchain-openai"], check=True)
#
# Boeing path (internal packages - adjust source as required by your org):
# subprocess.run([sys.executable, "-m", "pip", "install", "-q", "boeing-chat-model", "boeing-embeddings"], check=True)


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# ============================================================
# LLM PROVIDER SWITCH
# Set to "boeing" or "openai". This is the ONLY place you need
# to choose a provider - everything below uses `chat_client`
# regardless of which one is active.
# ============================================================
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "boeing").lower()   # "boeing" or "openai"

if LLM_PROVIDER == "boeing":
    # -------------------- BOEING PATH --------------------
    from boeing_chat_model import BoeingChatModel

    UDAL_PAT = os.getenv("UDAL_PAT")
    if not UDAL_PAT:
        raise ValueError(
            "UDAL_PAT not found in .env file (required when LLM_PROVIDER=boeing)."
        )

    chat_client = BoeingChatModel(
        udal_pat=UDAL_PAT,
        model="gpt-4o-mini",
        max_tokens=500,
        temperature=0,
    )
    print("[SETUP] LLM_PROVIDER=boeing -> using BoeingChatModel (gpt-4o-mini).")

elif LLM_PROVIDER == "openai":
    # -------------------- OPENAI PATH --------------------
    from langchain_openai import ChatOpenAI

    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    if not OPENAI_API_KEY:
        raise ValueError(
            "OPENAI_API_KEY not found in .env file (required when LLM_PROVIDER=openai)."
        )

    chat_client = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
        api_key=OPENAI_API_KEY,
    )
    print("[SETUP] LLM_PROVIDER=openai -> using ChatOpenAI (gpt-4o-mini).")

else:
    raise ValueError(f"Unknown LLM_PROVIDER '{LLM_PROVIDER}'. Use 'boeing' or 'openai'.")


---
## SECTION 1 - Sample Meeting Notes

Two examples: one where every action item has a clear owner, and one where an item is missing an owner - to demonstrate the guardrail.


In [ ]:
MEETING_NOTES_CLEAN = """
Weekly Platform Sync - July 8

Attendees: Priya, Sam, Jordan, Alex

Notes:
- Priya will update the API rate-limit docs by Friday.
- Sam is going to fix the flaky login test this week.
- Jordan needs to review the Q3 roadmap draft before next Monday's planning meeting.
- We agreed the staging environment naming convention is fine as-is, no action needed.
"""

MEETING_NOTES_MISSING_OWNER = """
Incident Retro - July 8

Attendees: Priya, Sam, Jordan, Alex

Notes:
- Someone needs to add a retry policy to the payment webhook handler, ideally by next Friday.
- Alex will write up the postmortem doc by end of day tomorrow.
- We should also look into why the alert fired 20 minutes late - no one picked this up yet.
"""

print("[FIXTURES] Loaded two sample meeting note sets.")


---
## SECTION 2 - Tools

Mocked here so the notebook runs offline. In a real deployment these would call an actual task tracker (Jira/Asana/Linear) and a real notification channel (Slack/email).


In [ ]:
# -- Tools (mocked) ------------------------------------------------------------------
_TASKS_DB = []

def create_task(title: str, owner: str, due_date: str) -> str:
    """Create a task for an action item and return its id."""
    task_id = f"TASK-{len(_TASKS_DB) + 1:04d}"
    _TASKS_DB.append({"id": task_id, "title": title, "owner": owner, "due_date": due_date})
    print(f"[TOOL: create_task] {task_id} | '{title}' -> {owner} (due {due_date})")
    return task_id


def send_notification(summary: str) -> bool:
    """Send a summary notification (mocked)."""
    print(f"[TOOL: send_notification]\n{summary}")
    return True


print("[TOOLS] Registered: create_task | send_notification")


---
## SECTION 3 - Guardrail: Never Guess an Owner

This check is plain Python, not an LLM judgment call. If notes don't name an owner, the agent must not invent one - it routes the item to a human instead of assigning it silently.


In [ ]:
# -- Guardrail: no owner guessing -----------------------------------------------------
NO_OWNER_VALUES = {"", "none", "null", "tbd", "unknown", "unassigned"}
DEFAULT_DEADLINE_DAYS = 7


def apply_owner_guardrail(items: list) -> tuple[list, list]:
    """Split extracted items into (assigned, unassigned). Never assigns a guessed owner."""
    assigned, unassigned = [], []
    for item in items:
        owner = (item.get("owner") or "").strip()
        if owner.lower() in NO_OWNER_VALUES:
            unassigned.append(item)
        else:
            assigned.append(item)
    return assigned, unassigned


def apply_default_deadline(item: dict) -> dict:
    """Fill in a one-week-out deadline if none was mentioned in the notes."""
    from datetime import datetime, timedelta
    deadline = (item.get("deadline") or "").strip()
    if deadline.lower() in {"", "none", "null", "tbd"}:
        item["deadline"] = (datetime.now() + timedelta(days=DEFAULT_DEADLINE_DAYS)).strftime("%Y-%m-%d")
        item["deadline_defaulted"] = True
    else:
        item["deadline_defaulted"] = False
    return item


print(f"[GUARDRAIL] apply_owner_guardrail active. Default deadline = {DEFAULT_DEADLINE_DAYS} days out.")


---
## SECTION 4 - Extraction Agent

A single-purpose LLM call that turns raw notes into structured action items. The prompt explicitly tells the model to use `null` rather than inventing a name or date - the guardrail above is the actual enforcement, but asking nicely first reduces how often it's needed.


In [ ]:
# -- Extraction agent ------------------------------------------------------------------
import json
import re
from langchain_core.messages import SystemMessage, HumanMessage

EXTRACTION_SYSTEM_PROMPT = """You are an assistant that extracts action items from raw meeting notes.

Return ONLY a JSON array, with no explanation and no markdown code fences.
Each element must be an object with exactly these keys:
  "task"     - a short description of the action item
  "owner"    - the person's name responsible, or null if no owner is named in the notes
  "deadline" - the deadline mentioned in the notes in any format, or null if none is mentioned

Do not guess an owner or a deadline that is not stated or clearly implied in the notes.
If the notes contain no action items, return an empty array: []
"""


def _strip_code_fences(text: str) -> str:
    return re.sub(r"^```(json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()


def extract_action_items(notes: str) -> list:
    """Call the LLM and parse its response into a list of action item dicts."""
    response = chat_client.invoke([
        SystemMessage(content=EXTRACTION_SYSTEM_PROMPT),
        HumanMessage(content=notes),
    ])
    raw = _strip_code_fences(response.content)
    try:
        items = json.loads(raw)
    except json.JSONDecodeError:
        print(f"[EXTRACTION] Could not parse model output as JSON:\n{raw}")
        items = []
    return items


print("[AGENT] extract_action_items ready.")


---
## SECTION 5 - Full Pipeline


In [ ]:
# -- Pipeline ---------------------------------------------------------------------------
def process_meeting_notes(notes: str) -> dict:
    print(f"\n{'='*70}\n[AGENT] Processing meeting notes\n{'='*70}")

    items = extract_action_items(notes)
    print(f"[EXTRACTION] {len(items)} action item(s) found.")

    assigned, unassigned = apply_owner_guardrail(items)
    if unassigned:
        print(f"[GUARDRAIL] {len(unassigned)} item(s) have no owner - routing to a human, not guessing.")

    created_tasks = []
    for item in assigned:
        item = apply_default_deadline(item)
        task_id = create_task(item["task"], item["owner"], item["deadline"])
        created_tasks.append({**item, "task_id": task_id})

    summary_lines = [
        f"{t['task_id']}: {t['task']} -> {t['owner']} (due {t['deadline']})"
        for t in created_tasks
    ]
    if unassigned:
        summary_lines.append(
            f"{len(unassigned)} action item(s) need a human to assign an owner: "
            + "; ".join(u["task"] for u in unassigned)
        )
    summary = "\n".join(summary_lines) if summary_lines else "No action items found in these notes."

    send_notification(summary)

    result = {
        "created_tasks": created_tasks,
        "unassigned_items": unassigned,
        "notification_summary": summary,
    }
    print(f"\n[RESULT] {len(created_tasks)} task(s) created | {len(unassigned)} unassigned")
    return result


print("[PIPELINE] process_meeting_notes ready.")


---
## SECTION 6 - Example Runs


In [ ]:
# Every action item has a named owner -> all tasks created, none flagged
result_clean = process_meeting_notes(MEETING_NOTES_CLEAN)
result_clean


In [ ]:
# One action item has no named owner -> guardrail flags it instead of guessing
result_missing_owner = process_meeting_notes(MEETING_NOTES_MISSING_OWNER)
result_missing_owner


---
## Summary

- **One agent**: `extract_action_items`, an LLM call constrained to return structured JSON.
- **Two tools**: `create_task`, `send_notification` - mocked, but the same shape as real integrations.
- **One guardrail**: `apply_owner_guardrail` - deterministic, not an LLM judgment call, so an ambiguous note can never result in a task silently assigned to the wrong person.
- **Provider switch**: `LLM_PROVIDER` in Section 0 selects Boeing or OpenAI; every line downstream is provider-agnostic.

A Flask + HTML version of this exact pipeline is provided alongside this notebook as `app.py`, `index.html`, and `requirements.txt`, for people who want to interact with the agent through a browser instead of a notebook.
